In [1]:
import numpy as np
import pandas as pd

In [2]:
import re
import nltk

In [3]:
from numpy import random
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import twitter_samples, stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, SimpleRNN, Dense, Dropout, Embedding

In [4]:
nltk.download('twitter_samples')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
stop_words = stopwords.words('english')

In [6]:
pos_tweets = twitter_samples.strings('positive_tweets.json')
neg_tweets = twitter_samples.strings('negative_tweets.json')

In [7]:
pos_tweets

['#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)',
 '@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!',
 '@DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?!',
 '@97sides CONGRATS :)',
 'yeaaaah yippppy!!!  my accnt verified rqst has succeed got a blue tick mark on my fb profile :) in 15 days',
 '@BhaktisBanter @PallaviRuhail This one is irresistible :)\n#FlipkartFashionFriday http://t.co/EbZ0L2VENM',
 "We don't like to keep our lovely customers waiting for long! We hope you enjoy! Happy Friday! - LWWF :) https://t.co/smyYriipxI",
 '@Impatientraider On second thought, there’s just not enough time for a DD :) But new shorts entering system. Sheep must be buying.',
 'Jgh , but we have to go to Bayan :D bye',
 'As an act of mischievousness, am calling the ETL layer of our in-house warehousing 

In [8]:
neg_tweets[2]

'@Hegelbon That heart sliding into the waste basket. :('

In [9]:
print(len(pos_tweets))

5000


In [10]:
print(len(neg_tweets))

5000


In [11]:
def clean_text(tweet):
  tweet = re.sub('(#|@)\w*', '', tweet)
  tweet = re.sub('(\?|!)+', '', tweet)
  tweet = re.sub('[:()\\\]', '', tweet)
  tweet = re.sub('^\s+', '', tweet)
  tweet = re.sub('\s+$', '', tweet)
  tweet = re.sub('(\.|\,)', '', tweet)
  tweet = re.sub('https?://\S+', '', tweet)
  tweet = re.sub('\s\d+\s', '', tweet)
  return tweet


<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\?'
<>:4: SyntaxWarning: invalid escape sequence '\]'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:8: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\?'
<>:4: SyntaxWarning: invalid escape sequence '\]'
<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:8: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3624484401.py:2: SyntaxWarning: invalid escape sequence '\w'
  tweet = re.sub('(#|@)\w*', '', tweet)
/tmp/ipython-input-3624484401.py:3: SyntaxWarning: invalid escape sequence '\?'


In [12]:
def preprocess_sentences(tweets):
  clean_tweets=[]
  for tweet in tweets:
    tweet=clean_text(tweet)
    tweet=nltk.word_tokenize(tweet)
    tweet=[w.lower() for w in tweet]
    tweet=[word for word in tweet if word not in stop_words]
    lemmatizer=WordNetLemmatizer()
    clean_tweet=[lemmatizer.lemmatize(word) for word in tweet]
    clean_tweets.append(clean_tweet)
  return clean_tweets

In [13]:
pos_tweets = preprocess_sentences(pos_tweets)

In [14]:
neg_tweets = preprocess_sentences(neg_tweets)

In [15]:
pos_labels = [1] * len(pos_tweets)

In [16]:
neg_labels = [0] * len(neg_tweets)

In [17]:
tweets = pos_tweets + neg_tweets

In [18]:
labels = pos_labels + neg_labels

In [19]:
zip_list = list(zip(tweets, labels))

In [20]:
zip_list[10]

(['top', 'influencers', 'community', 'week'], 1)

In [21]:
zip_list[8000]

(['sorna'], 0)

In [22]:
random.shuffle(zip_list)

In [23]:
tweets, labels = zip(*zip_list)

In [24]:
tweets[2]

['ate', 'whole', 'pizza', 'idk', 'feel']

In [25]:
labels[2]

0

In [26]:
df = pd.DataFrame({'tweets': tweets, 'labels':labels})

In [27]:
df.head(10)

,tweets,labels
0,"[``, zayn, '', start, small, letter, z]",0
1,"[babeeeee, 're, demn, hotaisndonwyvauwjoqhsjsn...",0
2,"[ate, whole, pizza, idk, feel]",0
3,"[thanks, -, someone, maybe, look, casefast, -,...",0
4,"[home, dormtel, near, st, scho, girl, siya, th...",0
5,"['ve, wanted, panda, likedays]",0
6,"[favorite, tea]",1
7,"[yes, vicky, omg, -]",0
8,"[happy, friday]",1
9,"['s, worst, pain]",0


**Text Representative**

In [28]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 88.3 MB/s eta 0:00:00


In [29]:
from gensim.models import Word2Vec

In [30]:
embedding_dim = 100
window_size = 5
sg = 0 # 0 for CBOW 1 for Skip-gram
min_count = 1
sentences = df['tweets'].tolist()

In [31]:
sentences

[['``', 'zayn', "''", 'start', 'small', 'letter', 'z'],
 ['babeeeee',
  "'re",
  'demn',
  'hotaisndonwyvauwjoqhsjsnaihsuswtf',
  'https//tco/kwwv5grny7'],
 ['ate', 'whole', 'pizza', 'idk', 'feel'],
 ['thanks',
  '-',
  'someone',
  'maybe',
  'look',
  'casefast',
  '-',
  'totally',
  'fucked',
  'mad',
  'expensive'],
 ['home',
  'dormtel',
  'near',
  'st',
  'scho',
  'girl',
  'siya',
  'tho',
  'want',
  'help',
  'find',
  'oneee'],
 ["'ve", 'wanted', 'panda', 'likedays'],
 ['favorite', 'tea'],
 ['yes', 'vicky', 'omg', '-'],
 ['happy', 'friday'],
 ["'s", 'worst', 'pain'],
 ['wonderful',
  'thing',
  "'s",
  'made',
  'feel',
  'fluffy',
  'inside',
  'great',
  'pirouette',
  'moose'],
 ['want', 'brace', 'back'],
 ['*',
  '✫',
  '✧',
  '˚',
  '✧',
  '·',
  '✵',
  '⊹',
  'thank',
  'always',
  'putting',
  'smile',
  'face',
  'mind',
  'following',
  'nice',
  'day',
  'x',
  '1707'],
 ['please',
  'follow',
  'waiting',
  'follow',
  'back',
  'please',
  'follow',
  'back',
 

In [32]:
model = Word2Vec(sentences, vector_size=embedding_dim, window=window_size, sg=sg, min_count=min_count)

In [33]:
def document_vectors(model, doc):
  return [model.wv[word] for word in doc]

In [34]:
df['tweets'] = df['tweets'].apply(lambda x: document_vectors(model, x))

In [35]:
df['tweets'].iloc[1]

[array([ 0.00415027,  0.0014872 ,  0.00412632, -0.00067252,  0.00919353,
         0.00096671,  0.00036127,  0.01471491,  0.00614631,  0.00121638,
         0.00080354, -0.01177797, -0.00340318, -0.00516651, -0.00414641,
        -0.000214  ,  0.00298084, -0.00460086, -0.00471242, -0.01427959,
         0.00116178, -0.00733068,  0.00023101,  0.00326876,  0.00015122,
         0.00521968, -0.01363712,  0.00653083, -0.00160385, -0.00732065,
         0.00710009, -0.00195488,  0.00210483, -0.01042295,  0.00207846,
         0.0048979 , -0.00926927, -0.01208362,  0.00169214, -0.01600923,
         0.00640771,  0.00423066, -0.00314085,  0.00167592,  0.01147762,
        -0.00622125,  0.00356056,  0.00896618, -0.00675201,  0.01253479,
        -0.00056403, -0.00039426, -0.00148897,  0.00165492, -0.00171604,
         0.01043702,  0.01139059,  0.00233077, -0.00504028,  0.00148173,
         0.00368032,  0.00732869,  0.00930143,  0.0091139 , -0.00477192,
        -0.00590173,  0.00023869, -0.00138155,  0.0

In [36]:
from keras.preprocessing.sequence import pad_sequences

In [37]:
x = df['tweets']
y = df['labels']

In [38]:
x = pad_sequences(x, maxlen=100, dtype='float32')

In [39]:
x

array([[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [-3.20885703e-02,  8.64903629e-02, -1.13545042e-02, ...,
         -1.12530716e-01,  1.66323576e-02, -7.73113174e-03],
        [-5.52018359e-03,  1.72190946e-02,  6.03292976e-03, ...,
         -9.98302083e-03, -4.31502936e-03,  5.97230485e-03],
        [-9.05849691e-03,  4.43145558e-02, -7.67171523e-03, ...,
         -5.00478409e-02, -7.75888475e-05,  1.00118271e-03]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [40]:
x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size=.2, random_state=42)

In [41]:
x_train.shape

(8000, 100, 100)

In [42]:
x_test.shape

(2000, 100, 100)

In [43]:
y_train.shape

(8000,)

In [44]:
y_test.shape

(2000,)

**Build Models**

SimpleRNN

In [91]:
model = Sequential()
model.add(SimpleRNN(128, return_sequences=True))
model.add(SimpleRNN(128))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [92]:
from keras.optimizers import Adam

In [93]:
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=.001), metrics=['accuracy'])

In [94]:
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 104ms/step - accuracy: 0.5113 - loss: 0.7120 - val_accuracy: 0.5100 - val_loss: 0.6874
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 96ms/step - accuracy: 0.5112 - loss: 0.6888 - val_accuracy: 0.5025 - val_loss: 0.6867
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 92ms/step - accuracy: 0.5170 - loss: 0.6858 - val_accuracy: 0.5030 - val_loss: 0.6877
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - accuracy: 0.5269 - loss: 0.6852 - val_accuracy: 0.5175 - val_loss: 0.6881
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.5287 - loss: 0.6871 - val_accuracy: 0.5035 - val_loss: 0.6904
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 90ms/step - accuracy: 0.5216 - loss: 0.6860 - val_accuracy: 0.5170 - val_loss: 0.6925
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.5181 - loss: 0.6924 - val_accuracy: 0.5300 - val_loss: 0.6907
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 21s 95ms/step - accuracy: 0.5327 - loss: 0.6880 -

In [98]:
loss, accuracy = model.evaluate(x_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.5366 - loss: 0.6815


In [99]:
print('Accuracy', accuracy)

Accuracy 0.5254999995231628


In [100]:
print('Loss', loss)

Loss 0.6847097277641296


LSTM

In [101]:
model = Sequential()
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(128))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [50]:
from keras.optimizers import Adam

In [103]:
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=.001), metrics=['accuracy'])

In [104]:
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 56s 382ms/step - accuracy: 0.5162 - loss: 0.6915 - val_accuracy: 0.5020 - val_loss: 0.6935
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 46s 367ms/step - accuracy: 0.5132 - loss: 0.6906 - val_accuracy: 0.4930 - val_loss: 0.6896
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 379ms/step - accuracy: 0.5198 - loss: 0.6871 - val_accuracy: 0.5240 - val_loss: 0.6894
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 379ms/step - accuracy: 0.5130 - loss: 0.6874 - val_accuracy: 0.5025 - val_loss: 0.6880
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 48s 380ms/step - accuracy: 0.5188 - loss: 0.6859 - val_accuracy: 0.5050 - val_loss: 0.6884
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 372ms/step - accuracy: 0.5283 - loss: 0.6831 - val_accuracy: 0.5040 - val_loss: 0.6883
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 378ms/step - accuracy: 0.5238 - loss: 0.6842 - val_accuracy: 0.5090 - val_loss: 0.6882
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 376ms/step - accuracy: 0.5241 - loss: 0

In [108]:
loss, accuracy = model.evaluate(x_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 178ms/step - accuracy: 0.5396 - loss: 0.6806


In [109]:
print('Accuracy', accuracy)

Accuracy 0.5414999723434448


In [110]:
print('Loss', loss)

Loss 0.684662401676178


**Bidirectional**

In [45]:
from keras.layers import Bidirectional

In [46]:
model = Sequential()
model.add(Bidirectional(LSTM(128, return_sequences=True), input_shape=(100, 100)))
model.add(LSTM(64))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [47]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [48]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.5)
]

In [51]:
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])

In [56]:
model.fit(x_train, y_train, validation_split=0.2, epochs=50, batch_size=32, callbacks=callbacks)

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.6163 - loss: 0.6380 - val_accuracy: 0.6056 - val_loss: 0.6398 - learning_rate: 5.0000e-04
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.6147 - loss: 0.6402 - val_accuracy: 0.6338 - val_loss: 0.6265 - learning_rate: 5.0000e-04
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.6262 - loss: 0.6333 - val_accuracy: 0.6137 - val_loss: 0.6407 - learning_rate: 5.0000e-04
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.6259 - loss: 0.6300 - val_accuracy: 0.6494 - val_loss: 0.6241 - learning_rate: 5.0000e-04
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6312 - loss: 0.6284 - val_accuracy: 0.6269 - val_loss: 0.6221 - learning_rate: 5.0000e-04
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.6299 - loss: 0.6260 - val_accuracy: 0.6356 - val_loss: 0.6255 - learning_rate: 5.0000e-04
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - acc

In [53]:
loss, accuracy = model.evaluate(x_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.6126 - loss: 0.6410


In [54]:
print('Accuracy', accuracy)

Accuracy 0.6039999723434448


In [55]:
print('Loss', loss)

Loss 0.6451241374015808
